# The three-number comparator -- summary figure

Pure plotting pass over cached results from `explore_multicompare.ipynb` (n3), `explore_multicompare2.ipynb` (n2), `explore_comparatorfig.ipynb`/`makefig_comparator.ipynb` (the two-number comparator neurons), and this session's L19/L20 read-out-token work. No model, no recomputation.

In [ ]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D            # noqa: F401  (registers the 3d projection)

RESULTS_DIR, FIGS_DIR = "results", "figs"
os.makedirs(FIGS_DIR, exist_ok=True)
REGIME = "2digit"

## Load cached results

In [ ]:
def load(name):
    return dict(np.load(f'{RESULTS_DIR}/{name}_{REGIME}.npz', allow_pickle=True))

# n3 -- explore_multicompare.ipynb
POOLD      = load('multicompare_pool')
UCOS       = load('multicompare_ucos')
DIRS3      = load('multicompare_dirs')
SPACE3     = load('multicompare_space')
RF3        = load('multicompare_rf')
ATTR3      = load('multicompare_attr')
DAS15_3    = load('multicompare_das15')
LATE       = load('multicompare_late')
LATEHEAD20 = load('multicompare_latehead20')
H27CLOUD   = load('multicompare_h27l20cloud')
PCA20      = load('multicompare_pca20causal')
TWO        = load('comparator_data')          # overlap_k20: the two-number comparator neurons

# n2 -- explore_multicompare2.ipynb
DIRS2  = load('multicompare2_dirs')
SPACE2 = load('multicompare2_space')
RF2    = load('multicompare2_rf')
DAS15_2 = load('multicompare2_das15')

T, POS = int(POOLD['T']), POOLD['pos'].tolist()
LAST_POS = T - 1
print(f'T={T}  POS={POS}  LAST_POS={LAST_POS}')

## Style

In [ ]:
LABEL_FS, TICK_FS, LEGEND_FS, TITLE_FS = 20, 14, 14, 15
FS_PANEL = 20
DPI, SAVE = 300, True

COL = {'full': '#2166AC', 'das': '#C97B00', 'pc1': '#C06A9B', 'h14': '#117777', 'ceil': '#6E6E6E'}
NCOL = {1: '#C97B00', 2: '#2166AC', 3: '#DDCC77'}   # y3 Tol sand: green belongs to the PRGn fields
Y = {1: r'$y_1$', 2: r'$y_2$', 3: r'$y_3$'}
CMAP = 'viridis'
ck = lambda s: str(s).replace(' ', '_').replace('>', 'gt')


def style_axes(ax, xlabel=None, ylabel=None, title=None, grid=True):
    if xlabel: ax.set_xlabel(xlabel, fontsize=LABEL_FS)
    if ylabel: ax.set_ylabel(ylabel, fontsize=LABEL_FS)
    if title:  ax.set_title(title, fontsize=TITLE_FS)
    ax.tick_params(labelsize=TICK_FS)
    if grid: ax.grid(alpha=0.3, axis='y')
    for side in ('top', 'right'):
        if side in ax.spines: ax.spines[side].set_visible(False)


def panel_letter(ax, s):
    ax.text(-0.16, 1.05, s, transform=ax.transAxes, fontsize=FS_PANEL,
            fontweight='bold', ha='left', va='bottom')


def savefig(fig, name):
    if SAVE:
        fig.savefig(f'{FIGS_DIR}/multicompare_{name}.png', dpi=DPI, bbox_inches='tight')


# neuron-pool bookkeeping, matching data_multicompare's NAME/POOL_LAYER/POOL_WITHIN exactly
D_FF, NEURON_LAYERS = 18944, [14, 15]
POOL_LAYER  = np.concatenate([np.full(D_FF, l) for l in NEURON_LAYERS])
POOL_WITHIN = np.concatenate([np.arange(D_FF) for _ in NEURON_LAYERS])
NAME = lambda p: f'L{POOL_LAYER[p]}#{POOL_WITHIN[p]}'


def rf_cube3(pooled_id):
    """(G, G, G) activation cube over (y1, y2, y3) for one pooled neuron, from RF3."""
    l, w = int(POOL_LAYER[pooled_id]), int(POOL_WITHIN[pooled_id])
    c = int(np.where(RF3[f'rf_idx_L{l}'] == w)[0][0])
    g = RF3['rf_grid']
    return RF3[f'rf_acts_L{l}'][:, c].reshape(len(g), len(g), len(g)), g


def rf_field2(pooled_id):
    """(G, G) activation field over (y1, y2) for one pooled neuron, from RF2."""
    l, w = int(POOL_LAYER[pooled_id]), int(POOL_WITHIN[pooled_id])
    c = int(np.where(RF2[f'rf_idx_L{l}'] == w)[0][0])
    g = RF2['rf_grid']
    return RF2[f'rf_acts_L{l}'][:, c].reshape(len(g), len(g)), g


# same diverging map as makefig_comparator's receptive fields: colourblind-safe, and not
# red/blue (already spoken for by the y1/y2/y3 palette elsewhere in this figure)
from matplotlib.colors import LinearSegmentedColormap


def trim_cmap(name, t, n=256):
    return LinearSegmentedColormap.from_list(
        f'{name}_t{t}', matplotlib.colormaps[name](np.linspace(t, 1 - t, n)))


RF_CM = trim_cmap('PRGn', 0.16)

## The figure

In [ ]:
import matplotlib.patches as mpatches

# ==================================================================================
# FONT SIZES  --  edit any of these
# ==================================================================================
FS_AXIS    = 26   # x / y axis labels
FS_TICK    = 20   # tick labels
FS_PANEL   = 31   # the bold (a) ... (f)
FS_LEGEND  = 19   # legend entries
FS_RF_AXIS = 20   # (c) axis labels on the receptive fields
FS_RF_NAME = 19   # (c) the neuron name over each row

# ==================================================================================
# LAYOUT / STYLE  --  edit any of these
# ==================================================================================
FIG_SIZE = (24.0, 13.0)
GRID_WSPACE, GRID_HSPACE = 0.30, 0.34
PANEL_LX, PANEL_LY = -0.16, 1.03
A_BINS = 9                  # (a) equal-width bins of y_j, one point per bin
D_HEADROOM = 1.45           # (d) y limit as a multiple of the tallest bin: labels sit above the data
# One meaning per colour: amber / blue / sand are ONLY y1 / y2 / y3. (f)'s rank-limited patch is
# teal, a hue used nowhere else here; full rank and "not max" are grey. The
# receptive fields use trimmed PRGn, as in makefig_comparator -- y3 is sand so its green is free.
RANKED_COL, FULL_COL = '#35978F', '0.72'     # (f): the 2-PC patch vs the full-rank patch
# (f)'s flag bars take the colour of the number perturbed in their experiment (y2 / y3), as in (d).
NOTMAX_COL = '0.35'                          # (d): the "not max" outline
RF_CM = trim_cmap('PRGn', 0.16)              # (c): green = positive, purple = negative

fig = plt.figure(figsize=FIG_SIZE)
gs = fig.add_gridspec(2, 3, wspace=GRID_WSPACE, hspace=GRID_HSPACE)

LATE_GROUPS = ['s1>s2', 's1>s3', 's2>s3']
GROUP_LAB = {'s1>s2': '$y_1{>}y_2$', 's1>s3': '$y_1{>}y_3$', 's2>s3': '$y_2{>}y_3$'}
ARGMAX_SLOT3 = DIRS3['cloud_y'].argmax(1)
BIGGER2 = (DIRS2['cloud_y'][:, 1] > DIRS2['cloud_y'][:, 0]).astype(int)   # 0: y1>y2, 1: y2>y1
V = {1: r'$\mathbf{v}_1$', 2: r'$\mathbf{v}_2$', 3: r'$\mathbf{v}_3$'}
se_of = lambda v: v.std() / np.sqrt(len(v))


def panel_letter(ax, s, x=PANEL_LX, y=PANEL_LY):
    ax.text(x, y, s, transform=ax.transAxes, fontsize=FS_PANEL, fontweight='bold',
            ha='left', va='bottom')


def big_axes(ax, xlabel=None, ylabel=None):
    style_axes(ax, grid=False)
    if xlabel: ax.set_xlabel(xlabel, fontsize=FS_AXIS)
    if ylabel: ax.set_ylabel(ylabel, fontsize=FS_AXIS)
    ax.tick_params(labelsize=FS_TICK)


# ================================================================ (a) linear codes at y3 =========
# Each direction's component on the 1500-prompt cloud at the position of y3, against the number it
# carries, in A_BINS equal-width bins (mean +- SE). v1 and v2 live in the outputs of heads L14.H14
# and L14.H18, v3 in the L13 residual, so each is z-scored to share one axis.
ax = fig.add_subplot(gs[0, 0])
yc = DIRS3['cloud_y']
for j, cloud in ((1, 'cloud_head14'), (2, 'cloud_head18'), (3, 'cloud_l13')):
    X = DIRS3[cloud].astype(np.float64)
    c = (X - X.mean(0)) @ DIRS3[f'v{j}'].astype(np.float64)
    yj = yc[:, j - 1].astype(float)
    c = (c - c.mean()) / c.std()
    c = c * (1 if np.corrcoef(c, yj)[0, 1] >= 0 else -1)          # signs are arbitrary
    edges = np.linspace(yj.min(), yj.max() + 1e-9, A_BINS + 1)
    b = np.digitize(yj, edges) - 1
    mid = np.array([yj[b == k].mean() for k in range(A_BINS)])
    m = np.array([c[b == k].mean() for k in range(A_BINS)])
    se = np.array([c[b == k].std(ddof=1) / np.sqrt((b == k).sum()) for k in range(A_BINS)])
    ax.errorbar(mid, m, yerr=se, fmt='-o', ms=8, lw=2.6, color=NCOL[j], elinewidth=1.6,
                capsize=3, label=f'{V[j]} vs {Y[j]}')
big_axes(ax, xlabel='$y_j$', ylabel=r'comp. along $\mathbf{v}_j$ (z-score)')
ax.legend(fontsize=FS_LEGEND, frameon=False, loc='upper left', handletextpad=0.4)
panel_letter(ax, '(a)')

# ================================================================ (b) causal effect at y3 ========
ax = fig.add_subplot(gs[0, 1])
CASE_LIST3 = ['y1', 'y2', 'y3']
ARMS1B = [('v1 only', V[1]), ('v2 only', V[2]), ('v3 only', V[3]),
          ('span preMLP', 'span\n(' + ','.join(V[k] for k in (1, 2, 3)) + ')')]
x, w = np.arange(len(ARMS1B)), 0.8 / len(CASE_LIST3)
for ci, case in enumerate(CASE_LIST3):
    j = int(case[1])
    vals = [SPACE3[f'sp_posrec_{case}_{ck(nm)}'] for nm, _ in ARMS1B]
    xp = x + (ci - (len(CASE_LIST3) - 1) / 2) * w
    ax.bar(xp, [v.mean() for v in vals], w, yerr=[se_of(v) for v in vals], capsize=3,
           color=NCOL[j], alpha=0.9, label=f'{Y[j]} perturbed')
ax.set_xticks(x); ax.set_xticklabels([lab for _, lab in ARMS1B])
ax.set_ylim(-0.1, 1.2)
big_axes(ax, ylabel='PR')
ax.legend(fontsize=FS_LEGEND, frameon=False, loc='upper left', handlelength=1.2, handletextpad=0.4)
panel_letter(ax, '(b)')

# ================================================================ (c) receptive fields at y3 =====
LOCAL_ID, GLOBAL_ID = 5076, D_FF + 12784   # L14#5076, L15#12784
NEURONS_C1 = [(LOCAL_ID, 'MLP14 #5076'), (GLOBAL_ID, 'MLP15 #12784')]
MARGIN_TRIOS = [(2, '$y_1$', '$y_2$'), (1, '$y_1$', '$y_3$'), (0, '$y_2$', '$y_3$')]
inner = gs[0, 2].subgridspec(2, 3, wspace=0.28, hspace=0.42)
for ri, (pooled, label) in enumerate(NEURONS_C1):
    cube, g3 = rf_cube3(pooled)
    ext3 = [g3[0], g3[-1], g3[0], g3[-1]]
    for ci, (avg_axis, xl, yl) in enumerate(MARGIN_TRIOS):
        ax = fig.add_subplot(inner[ri, ci])
        f_ = cube.mean(axis=avg_axis)
        ax.imshow((f_ / (np.abs(f_).max() + 1e-9)).T, origin='lower', extent=ext3, cmap=RF_CM,
                  vmin=-1, vmax=1, aspect='equal')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(xl, fontsize=FS_RF_AXIS, labelpad=2)
        ax.set_ylabel(yl, fontsize=FS_RF_AXIS, labelpad=2)
        if ci == 1:          # the neuron's name over its row
            ax.set_title(label, fontsize=FS_RF_NAME, pad=6)
        if ri == 0 and ci == 0:
            panel_letter(ax, '(c)', x=-0.55, y=1.10)

# ================================================================ (d) the two flags, stacked =====
# The L15 rank-1 DAS component on the cloud, at the position of y2 (top) and of y3 (bottom). The
# number that is the max is filled in its own colour; "not max" is a grey outline drawn over it, so
# nothing blends where the two overlap.
dsub = gs[1, 0].subgridspec(2, 1, hspace=0.12)
FLAGS = [(DAS15_2['proj_y2_k1'][:, 0], BIGGER2 == 1, 2),
         (DAS15_3['proj_y3_k1'][:, 0], ARGMAX_SLOT3 == 2, 3)]
lo = min(p.min() for p, _, _ in FLAGS); hi = max(p.max() for p, _, _ in FLAGS)
bins = np.linspace(lo, hi, 45)
for k, (proj, is_max, j) in enumerate(FLAGS):
    ax = fig.add_subplot(dsub[k, 0])
    ax.hist(proj[is_max], bins=bins, color=NCOL[j], alpha=0.85, label=f'{Y[j]} max')
    ax.hist(proj[~is_max], bins=bins, histtype='step', color=NOTMAX_COL, lw=1.8,
            label=f'{Y[j]} not max')
    ax.set_xlim(lo, hi); ax.set_ylim(0, D_HEADROOM * ax.get_ylim()[1])
    big_axes(ax, xlabel='flag' if k == 1 else None, ylabel='count')
    ax.yaxis.label.set_size(FS_AXIS - 4)
    ax.text(0.02, 0.93, f'at {Y[j]}', transform=ax.transAxes, fontsize=FS_LEGEND, va='top')
    if k == 0:
        ax.set_xticklabels([])
        panel_letter(ax, '(d)', y=1.06)
    ax.legend(fontsize=FS_LEGEND - 2, frameon=False, loc='upper right', ncol=2, handlelength=1.2,
              columnspacing=1.0, borderaxespad=0.2)

# ================================================================ (e) the answer at the last token
ax = fig.add_subplot(gs[1, 1])
MAXSLOT_LATE = LATE['late_y'].argmax(1) + 1
pcL20 = LATE['pca_coords_L20']
# y3's sand is pale on white, so its points are drawn opaque; the other two keep alpha 0.65.
for slot in (1, 2, 3):
    m_ = MAXSLOT_LATE == slot
    ax.scatter(pcL20[m_, 0], pcL20[m_, 1], color=NCOL[slot], s=12, edgecolors='none',
               alpha=1.0 if slot == 3 else 0.65)
big_axes(ax, xlabel='PC1 (L20, last token)', ylabel='PC2')
for slot in (1, 2, 3):
    ax.scatter([], [], color=NCOL[slot], s=60, label=f'max $y_{slot}$')
ax.legend(fontsize=FS_LEGEND, frameon=False, loc='lower left', handletextpad=0.2)
panel_letter(ax, '(e)')

# ================================================================ (f) rank-limited vs full patches
# The causal checks the old insets made, on one axis: each flag's rank-1 DAS direction alone, and
# the two PCs of (e) against the full L20 residual at the last token, in the three value orders.
# The flags have no full-rank bar (dropped from the old insets).
ax = fig.add_subplot(gs[1, 2])
PR_GROUPS = ([(f'flag\nat {Y[2]}', DAS15_2['posrec_y2_from_y2_k1'], None),
              (f'flag\nat {Y[3]}', DAS15_3['posrec_y3_from_y3_k1'], None)] +
             [(f'answer\n{GROUP_LAB[g]}', PCA20[f'posrec_{ck(g)}'], LATE[f'resid_posrec_{ck(g)}_L20'])
              for g in LATE_GROUPS])
xg, wg = np.arange(len(PR_GROUPS)), 0.38
# A group with no full-rank bar has its ranked bar centred on the tick.
xr = np.array([x_ if f_ is None else x_ - wg / 2 for x_, (_, _, f_) in zip(xg, PR_GROUPS)])
from matplotlib.legend_handler import HandlerTuple
# Flag bars in their perturbed number's colour (y2, y3); the 2-PC answer bars teal.
_bar_col = [NCOL[2], NCOL[3]] + [RANKED_COL] * len(LATE_GROUPS)
_bars = ax.bar(xr, [r_.mean() for _, r_, _ in PR_GROUPS], wg,
               yerr=[se_of(r_) for _, r_, _ in PR_GROUPS], capsize=3, color=_bar_col)
_full = [(x_ + wg / 2, f_) for x_, (_, _, f_) in zip(xg, PR_GROUPS) if f_ is not None]
ax.bar([x_ for x_, _ in _full], [f_.mean() for _, f_ in _full], wg,
       yerr=[se_of(f_) for _, f_ in _full], capsize=3, color=FULL_COL, label='full')
ax.set_xticks(xg); ax.set_xticklabels([lab for lab, _, _ in PR_GROUPS])
big_axes(ax, ylabel='PR')
ax.tick_params(axis='x', labelsize=FS_TICK - 3)
_fullh = ax.containers[-1]
ax.legend([_bars[2], _fullh], ['L20 PC1,2', 'full'],
          handler_map={tuple: HandlerTuple(ndivide=None, pad=0)},
          fontsize=FS_LEGEND, frameon=False, loc='upper right', handlelength=1.6)
panel_letter(ax, '(f)')

savefig(fig, 'summary')
# plt.savefig('./figs_mainpaper/multicompare_fig.pdf', dpi=300, bbox_inches='tight')
plt.show()

print('(f) PR, ranked / full:')
for lab, r_, f_ in PR_GROUPS:
    print(f"   {lab.replace(chr(10), ' '):>22}: {r_.mean():.3f}" + ('' if f_ is None else f' / {f_.mean():.3f}'))
